In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
%cd /content/drive/My Drive

/content/drive/My Drive


In [28]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters
!pip install -q chromadb pypdf beautifulsoup4 sentence-transformers faiss-cpu
!pip install -q openai python-dotenv

In [29]:
# Import libraries
import os
import numpy as np
from getpass import getpass

# Set OpenAI API key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✓ Setup complete!")

✓ Setup complete!


In [30]:
from langchain_community.document_loaders import CSVLoader

# Import medreason instruction dataset
loader = CSVLoader('medreason-instruction-dataset.csv')
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"\nTotal characters: {len(docs[0].page_content):,}")
print(f"\nFirst 500 characters:\n{docs[0].page_content[:500]}...")
print(f"\nMetadata: {docs[0].metadata}")

Loaded 31535 document(s)

Total characters: 67

First 500 characters:
query: Most sensitive test for H pylori
answer: D. Urea breath test...

Metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0}


Split documents into smaller chunks for embedding and retrieval.
RecursiveCharacterTextSplitter

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,            # Maximum chunk size (characters)
    chunk_overlap=200,          # Overlap between chunks
    add_start_index=True,       # Track position in original document
    separators=["\n\n", "\n", " ", ""]  # Try separators in order
)

# Split the loaded documents
all_splits = text_splitter.split_documents(docs)

print(f"Split each document into {len(all_splits)} chunks")
print(f"\nChunk 0 length: {len(all_splits[0].page_content)} characters")
print(f"Chunk 0 metadata: {all_splits[0].metadata}")
print(f"\nFirst chunk content:\n{all_splits[0].page_content}")

Split each document into 32048 chunks

Chunk 0 length: 67 characters
Chunk 0 metadata: {'source': 'medreason-instruction-dataset.csv', 'row': 0, 'start_index': 0}

First chunk content:
query: Most sensitive test for H pylori
answer: D. Urea breath test


Check Chunk Overlap

In [32]:
# Demonstrate overlap with simple example
simple_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

sample_text = """Artificial intelligence is transforming how we interact with technology.
Machine learning models can now understand and generate human language with remarkable accuracy.
This has enabled new applications in search, customer service, and content creation."""

simple_chunks = simple_splitter.split_text(sample_text)

print(f"Created {len(simple_chunks)} chunks with overlap:\n")
for i, chunk in enumerate(simple_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")
    print(f"{'-'*80}")

Created 7 chunks with overlap:

Chunk 1 (46 chars): Artificial intelligence is transforming how we
--------------------------------------------------------------------------------
Chunk 2 (32 chars): how we interact with technology.
--------------------------------------------------------------------------------
Chunk 3 (46 chars): Machine learning models can now understand and
--------------------------------------------------------------------------------
Chunk 4 (43 chars): and generate human language with remarkable
--------------------------------------------------------------------------------
Chunk 5 (9 chars): accuracy.
--------------------------------------------------------------------------------
Chunk 6 (44 chars): This has enabled new applications in search,
--------------------------------------------------------------------------------
Chunk 7 (47 chars): search, customer service, and content creation.
---------------------------------------------------------------------

Compare Different Splitters

In [33]:
from langchain_text_splitters import CharacterTextSplitter

# CharacterTextSplitter (splits only on specified separator)
char_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="\n\n"  # Only split on double newlines
)

char_splits = char_splitter.split_documents(docs)

print(f"RecursiveCharacterTextSplitter: {len(all_splits)} chunks")
print(f"CharacterTextSplitter: {len(char_splits)} chunks")
print(f"\nRecursive splitter creates more uniform chunks by trying multiple separators.")

RecursiveCharacterTextSplitter: 32048 chunks
CharacterTextSplitter: 31758 chunks

Recursive splitter creates more uniform chunks by trying multiple separators.


Create embeddings and store them in a vector database.Create a vector store with Chroma.

In [34]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize embedding model
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Create vector store from document chunks
persist_directory = './chroma_db'

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print(f"✓ Vector store created with {vectordb._collection.count()} document chunks")
print(f"✓ Persisted to: {persist_directory}")

✓ Vector store created with 64345 document chunks
✓ Persisted to: ./chroma_db


Load existing vector store.

In [35]:
# Load existing vector store (no need to re-embed)
vectordb_loaded = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print(f"✓ Loaded vector store with {vectordb_loaded._collection.count()} documents")

✓ Loaded vector store with 64345 documents


Retrieval: Basic similarity search

In [36]:
question = "What is the most common site of origin of thrombotic pulmonary emboli?"

# Retrieve top-k similar documents
docs_retrieved = vectordb.similarity_search(question, k=3)

print(f"Query: {question}")
print(f"\nRetrieved {len(docs_retrieved)} documents:\n")

for i, doc in enumerate(docs_retrieved):
    print(f"{'='*80}")
    print(f"Document {i+1}")
    print(f"{'='*80}")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Start Index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"\nContent:\n{doc.page_content[:300]}...\n")

Query: What is the most common site of origin of thrombotic pulmonary emboli?

Retrieved 3 documents:

Document 1
Source: MedReason.csv
Start Index: 0

Content:
query: What is the most common site of origin of thrombotic pulmonary emboli?
answer: A. Deep leg veins...

Document 2
Source: medreason-instruction-dataset.csv
Start Index: 0

Content:
query: What is the most common site of origin of thrombotic pulmonary emboli?
answer: A. Deep leg veins...

Document 3
Source: medreason-instruction-dataset.csv
Start Index: 0

Content:
query: What is the most common site of origin of thrombotic pulmonary emboli?
answer: A. Deep leg veins...



Similarity search with scores

In [51]:
# Get documents with similarity scores
docs_with_scores = vectordb.similarity_search_with_score(question, k=3)

print(f"Query: {question}\n")

for i, (doc, score) in enumerate(docs_with_scores):
    print(f"Document {i+1} - Similarity Score: {score:.4f}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"{'-'*80}\n")

Query: Which class of antifungals inhibits the synthesis of the fungal cell wall?

Document 1 - Similarity Score: 0.3826
Content preview: query: Which class of antifungals inhibits the synthesis of the fungal cell wall?
answer: B. Echinocandins...
--------------------------------------------------------------------------------

Document 2 - Similarity Score: 0.3827
Content preview: query: Which class of antifungals inhibits the synthesis of the fungal cell wall?
answer: B. Echinocandins...
--------------------------------------------------------------------------------

Document 3 - Similarity Score: 0.3827
Content preview: query: Which class of antifungals inhibits the synthesis of the fungal cell wall?
answer: B. Echinocandins...
--------------------------------------------------------------------------------



Maximum Marginal Relevance (MMR)

In [52]:
question_mmr = "Which lesion displays an ill-defined border?"

print(f"Query: {question_mmr}\n")
print("="*80)
print("STANDARD SIMILARITY SEARCH")
print("="*80)

docs_ss = vectordb.similarity_search(question_mmr, k=3)
for i, doc in enumerate(docs_ss):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n" + "="*80)
print("MMR SEARCH (Diverse Results)")
print("="*80)

docs_mmr = vectordb.max_marginal_relevance_search(
    question_mmr,
    k=3,
    fetch_k=20  # Fetch 20 candidates, return 3 diverse ones
)

for i, doc in enumerate(docs_mmr):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n💡 Note: MMR results should be more diverse than similarity search.")

Query: Which lesion displays an ill-defined border?

STANDARD SIMILARITY SEARCH

Doc 1: query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

Doc 2: query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

Doc 3: query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

MMR SEARCH (Diverse Results)

Doc 1: query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

Doc 2: query: Oblitration of left cardiac shadow on PA view is due to:
answer: A. Lingular lesion...

Doc 3: query: Which hepatic lesion can be diagnosed with high accuracy using nuclear imaging?
answer: Hepatic hemangioma...

💡 Note: MMR results should be more diverse than similarity search.


Metadata Filtering: Filter results based on metadata.

In [53]:
# First, let's see what metadata is available
sample_doc = all_splits[0]
print("Available metadata fields:")
for key, value in sample_doc.metadata.items():
    print(f"  {key}: {value}")

# Example: Filter by source (if you have multiple sources)
# Note: This example uses the web source we loaded
source_filter = sample_doc.metadata.get('source')

print(f"\nFiltering by source: {source_filter}")

docs_filtered = vectordb.similarity_search(
    "Which of the following scoring system is used to see chest involvement in Sarcoidosis?",
    k=3,
    filter={"source": source_filter}
)

print(f"\nRetrieved {len(docs_filtered)} filtered documents")
for i, doc in enumerate(docs_filtered):
    print(f"Doc {i+1} source: {doc.metadata.get('source')}")

Available metadata fields:
  source: medreason-instruction-dataset.csv
  row: 0
  start_index: 0

Filtering by source: medreason-instruction-dataset.csv

Retrieved 3 filtered documents
Doc 1 source: medreason-instruction-dataset.csv
Doc 2 source: medreason-instruction-dataset.csv
Doc 3 source: medreason-instruction-dataset.csv


Question Answering with RAG

Combine retrieval with LLM generation.

RAG with Langchain agent

In [40]:
# No need for langchain_classic - we'll use the modern API
# Just ensure we have the latest langchain packages
!pip install -q --upgrade langchain langchain-openai langchain-community

In [54]:
from langchain.agents import create_agent
from langchain.tools import tool

# Create retrieval tool
@tool
def retrieve_context(query: str) -> str:
    """Retrieve information from the knowledge base to help answer questions."""
    print(f"Retrieving context for query: {query}")
    retrieved_docs = vectordb.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized

# Create agent with tools
agent = create_agent(
    model="gpt-4.1-nano",
    tools=[retrieve_context],
    #temperature=0,
    system_prompt="You are a helpful AI assistant. Use the retrieve_context tool to find information from the knowledge base when needed to answer questions accurately."
)

print("✓ RAG Agent created successfully!")

✓ RAG Agent created successfully!


In [55]:
# Ask a question using the agent
question = "Which of the following is not used for thrombo prophylaxis?"

print(f"Question: {question}\n")
print("="*80)

# Invoke the agent
result = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

print("\n" + "="*80)
print("FINAL ANSWER:")
print("="*80)
# The final message contains the agent's response
print(result["messages"][-1].content)

Question: Which of the following is not used for thrombo prophylaxis?

Retrieving context for query: Thrombo prophylaxis methods
Retrieving context for query: Medications used for thrombo prophylaxis

FINAL ANSWER:
Based on the retrieved information, the medication that is not used for thrombo prophylaxis is Antithrombin III.


Streaming agent responses

In [57]:
# Stream the agent's reasoning and responses
question = "Which of the following amino acid can produce oxaloacetate directly in a single reaction?"

print(f"Question: {question}\n")
print("="*80)
print("STREAMING AGENT EXECUTION:")
print("="*80 + "\n")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values"
):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]

    # Print different message types
    if hasattr(latest_message, 'content') and latest_message.content:
        if latest_message.type == "ai":
            print(f"\n🤖 Agent: {latest_message.content}")
    elif hasattr(latest_message, 'tool_calls') and latest_message.tool_calls:
        print(f"\n🔧 Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

print("\n" + "="*80)

Question: Which of the following amino acid can produce oxaloacetate directly in a single reaction?

STREAMING AGENT EXECUTION:


🔧 Calling tools: ['retrieve_context']
Retrieving context for query: amino acids that can produce oxaloacetate directly in a single reaction

🤖 Agent: The amino acid that can produce oxaloacetate directly in a single reaction is Aspartate.



RAG with custom prompt

In [58]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Initialize the LLM for the RAG chain
llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

# Custom prompt template
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.

Context: {context}

Question: {question}

Helpful Answer:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# Create RAG chain using LCEL (LangChain Expression Language)
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {
        "context": lambda x: format_docs(vectordb.similarity_search(x["question"], k=3)),
        "question": lambda x: x["question"]
    }
    | QA_CHAIN_PROMPT
    | llm
    | StrOutputParser()
)

# Helper function for RAG
def rag_qa(question):
    # Retrieve documents
    docs = vectordb.similarity_search(question, k=3)

    # Generate answer using the chain
    answer = rag_chain.invoke({"question": question})

    return answer, docs

# Test the custom RAG
question = "Which class of antifungals inhibits the synthesis of the fungal cell wall?"
answer, sources = rag_qa(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}\n")
print(f"\nSources ({len(sources)} documents):")
for i, doc in enumerate(sources):
    print(f"  {i+1}. {doc.metadata.get('source', 'Unknown')}")

Question: Which class of antifungals inhibits the synthesis of the fungal cell wall?

Answer: B. Echinocandins inhibit the synthesis of the fungal cell wall.


Sources (3 documents):
  1. MedReason.csv
  2. medreason-instruction-dataset.csv
  3. medreason-instruction-dataset.csv


Interactive RAG & QA

In [59]:
# Function for interactive Q&A with source display
def interactive_rag(question, show_sources=True, k=3):
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")

    # Retrieve
    docs = vectordb.similarity_search(question, k=k)

    # Generate using the RAG chain
    answer = rag_chain.invoke({"question": question})

    print(f"Answer: {answer}\n")

    if show_sources:
        print(f"{'='*80}")
        print(f"Retrieved Sources ({len(docs)} documents):")
        print(f"{'='*80}")
        for i, doc in enumerate(docs):
            print(f"\n[{i+1}] {doc.page_content[:200]}...")
            print(f"    Source: {doc.metadata.get('source', 'Unknown')}")

# Try multiple questions
questions = [
    "Which cyclooxygenase plays a role in maintaining GI mucosal integrity?",
    "Which of the following is true about hypothermia during anesthesia?",
    "Which lesion displays an ill-defined border?"
]

for q in questions:
    interactive_rag(q, show_sources=True)


Question: Which cyclooxygenase plays a role in maintaining GI mucosal integrity?

Answer: A. Cyclooxygenase 1

Retrieved Sources (3 documents):

[1] query: Which cyclooxygenase plays a role in maintaining GI mucosal integrity?
answer: A. Cyclooxygenase 1...
    Source: medreason-instruction-dataset.csv

[2] query: Which cyclooxygenase plays a role in maintaining GI mucosal integrity?
answer: A. Cyclooxygenase 1...
    Source: medreason-instruction-dataset.csv

[3] query: Which cyclooxygenase plays a role in maintaining GI mucosal integrity?
answer: A. Cyclooxygenase 1...
    Source: MedReason.csv

Question: Which of the following is true about hypothermia during anesthesia?

Answer: B. Prevented by giving warm fluids

Retrieved Sources (3 documents):

[1] query: Which of the following is true about hypothermia during anesthesia?
answer: B. Prevented by giving warm fluids...
    Source: MedReason.csv

[2] query: Which of the following is true about hypothermia during anesthesia?
answer

Evaluation: Retrieval Quality Metrics

In [46]:
# Create a simple test set
# In practice, you'd have ground truth labels

def evaluate_retrieval(query, k=5):
    """
    Evaluate retrieval quality.
    Note: This is a simplified version. In practice, you'd have ground truth.
    """
    # Retrieve documents
    docs = vectordb.similarity_search_with_score(query, k=k)

    print(f"Query: {query}")
    print(f"Retrieved {len(docs)} documents:\n")

    for i, (doc, score) in enumerate(docs):
        print(f"Rank {i+1} - Score: {score:.4f}")
        print(f"Content: {doc.page_content[:100]}...")
        print(f"{'-'*60}\n")

    # Calculate score statistics
    scores = [score for _, score in docs]
    print(f"\nScore Statistics:")
    print(f"  Mean: {np.mean(scores):.4f}")
    print(f"  Std:  {np.std(scores):.4f}")
    print(f"  Min:  {np.min(scores):.4f}")
    print(f"  Max:  {np.max(scores):.4f}")

# Test retrieval quality
evaluate_retrieval("Which flexor muscle is attached to hook of hamate?")

Query: Which flexor muscle is attached to hook of hamate?
Retrieved 5 documents:

Rank 1 - Score: 0.3208
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 2 - Score: 0.3208
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 3 - Score: 0.3209
Content: query: Which flexor muscle is attached to hook of hamate?
answer: C. Flexor digiti minimi...
------------------------------------------------------------

Rank 4 - Score: 0.8257
Content: query: The muscle of hand that contains a sesamoid bone is
answer: A. Flexor pollicis brevis...
------------------------------------------------------------

Rank 5 - Score: 0.8257
Content: query: The muscle of hand that contains a sesamoid bone is
answer: A. Flexor pollicis brevis...
---------------------------------------

Compare Retrieval Strategies

In [47]:
def compare_retrieval_methods(query, k=3):
    """
    Compare different retrieval strategies.
    """
    print(f"Query: {query}\n")

    # Method 1: Similarity Search
    print("="*80)
    print("METHOD 1: Similarity Search")
    print("="*80)
    docs_sim = vectordb.similarity_search(query, k=k)
    for i, doc in enumerate(docs_sim):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Method 2: MMR
    print("\n" + "="*80)
    print("METHOD 2: Maximum Marginal Relevance (MMR)")
    print("="*80)
    docs_mmr = vectordb.max_marginal_relevance_search(query, k=k, fetch_k=20)
    for i, doc in enumerate(docs_mmr):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Check overlap
    overlap = sum(1 for d1 in docs_sim if any(
        d1.page_content == d2.page_content for d2 in docs_mmr
    ))

    print(f"\n{'='*80}")
    print(f"Overlap: {overlap}/{k} documents are the same")
    print(f"MMR provides {'more' if overlap < k else 'similar'} diversity")

compare_retrieval_methods("Which lesion displays an ill-defined border?")

Query: Which lesion displays an ill-defined border?

METHOD 1: Similarity Search

[1] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[2] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[3] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

METHOD 2: Maximum Marginal Relevance (MMR)

[1] query: Which lesion displays an ill-defined border?
answer: B. Sclerosing osteitis...

[2] query: Oblitration of left cardiac shadow on PA view is due to:
answer: A. Lingular lesion...

[3] query: Which hepatic lesion can be diagnosed with high accuracy using nuclear imaging?
answer: Hepatic hemangioma...

Overlap: 3/3 documents are the same
MMR provides similar diversity


Answer Quality Assessment

In [48]:
# Simple answer quality check using LLM
from langchain_core.prompts import PromptTemplate

eval_template = """Evaluate the following answer based on the provided context.

Context: {context}

Question: {question}

Answer: {answer}

Evaluation Criteria:
1. Faithfulness: Is the answer supported by the context? (Yes/No)
2. Relevance: Does the answer address the question? (Yes/No)
3. Completeness: Is the answer complete? (Yes/No)

Provide your evaluation:"""

eval_prompt = PromptTemplate.from_template(eval_template)
eval_chain = eval_prompt | llm | StrOutputParser()

def evaluate_answer(question):
    # Generate answer
    answer, docs = rag_qa(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    print(f"Question: {question}")
    print(f"\nAnswer: {answer}")

    # Evaluate
    evaluation = eval_chain.invoke({
        "context": context,
        "question": question,
        "answer": answer
    })

    print(f"\n{'='*80}")
    print("EVALUATION:")
    print(f"{'='*80}")
    print(evaluation)

evaluate_answer("Which lesion displays an ill-defined border?")

Question: Which lesion displays an ill-defined border?

Answer: B. Sclerosing osteitis

EVALUATION:
1. Faithfulness: No — The context does not specify that sclerosing osteitis displays an ill-defined border. Repeatedly stating the same answer without supporting information does not confirm its correctness.

2. Relevance: Yes — The answer directly responds to the question about which lesion has an ill-defined border.

3. Completeness: No — The answer provides only the letter and lesion name without explanation or additional details.

Overall, the answer is not fully supported by the provided context and lacks completeness.


Trying different chuck sizes

In [49]:
# Test different chunk sizes
chunk_sizes = [500, 1000, 1500]

print("Testing different chunk sizes:\n")

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.2),  # 20% overlap
        add_start_index=True
    )

    splits = splitter.split_documents(docs)

    print(f"Chunk size: {size}")
    print(f"  Total chunks: {len(splits)}")
    print(f"  Avg chunk length: {np.mean([len(s.page_content) for s in splits]):.1f}")
    print(f"  Min chunk length: {min([len(s.page_content) for s in splits])}")
    print(f"  Max chunk length: {max([len(s.page_content) for s in splits])}")
    print()

print("💡 Smaller chunks → more granular retrieval but may lose context")
print("💡 Larger chunks → more context but less precise retrieval")

Testing different chunk sizes:

Chunk size: 500
  Total chunks: 35056
  Avg chunk length: 194.6
  Min chunk length: 3
  Max chunk length: 500

Chunk size: 1000
  Total chunks: 32048
  Avg chunk length: 209.7
  Min chunk length: 9
  Max chunk length: 1000

Chunk size: 1500
  Total chunks: 31669
  Avg chunk length: 211.7
  Min chunk length: 9
  Max chunk length: 1500

💡 Smaller chunks → more granular retrieval but may lose context
💡 Larger chunks → more context but less precise retrieval


Alternative Embedding Models: HuggingFace Embeddings

In [50]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use a free open-source embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embeddings
test_text = "This is a test sentence."
hf_embedding = hf_embeddings.embed_query(test_text)

print(f"HuggingFace Embedding dimension: {len(hf_embedding)}")
print(f"First 10 values: {hf_embedding[:10]}")

# Create vector store with HuggingFace embeddings
vectordb_hf = Chroma.from_documents(
    documents=all_splits[:50],  # Use subset for faster demo
    embedding=hf_embeddings,
    persist_directory='./chroma_db_hf'
)

print(f"\n✓ Created vector store with HuggingFace embeddings")
print(f"✓ Contains {vectordb_hf._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFace Embedding dimension: 384
First 10 values: [0.08429647237062454, 0.057953689247369766, 0.004493385087698698, 0.10582111030817032, 0.007083410397171974, -0.017844678834080696, -0.016888074576854706, -0.015228300355374813, 0.0404730923473835, 0.033422548323869705]

✓ Created vector store with HuggingFace embeddings
✓ Contains 150 documents
